<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/fundamentos/notebooks/c1_l4.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C1-L4 · Sharpe, Sortino y drawdown
Mide el camino del equity de SOL: retorno por unidad de riesgo y el peor susto desde el pico.

In [ ]:
import pandas as pd
from pathlib import Path

CSV = 'c1_l4_sol_equity.csv'
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/fundamentos/data/' + CSV
try:
    df = pd.read_csv(URL, parse_dates=["fecha"])
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data') / CSV, Path('data') / CSV, Path(CSV)]:
        if cand.exists():
            df = pd.read_csv(cand, parse_dates=["fecha"])
            break
    print('Fuente: local')
r = df["retorno_pct"] / 100
print(df.shape, "de", df["fecha"].min().date(), "a", df["fecha"].max().date())
print(df.head())

## Sharpe: retorno por unidad de susto
Media entre desviación, anualizado con √252. Por encima de 1.0 aceptable, por encima de 2.0 muy bueno.

In [ ]:
sharpe = r.mean() / r.std(ddof=0) * (252 ** 0.5)
print(f"Sharpe anualizado: {sharpe:.2f}")

## Sortino: solo castiga lo malo
Divide entre la volatilidad de los días malos. Si supera con holgura al Sharpe, hubo más fiestas que sustos.

In [ ]:
down = r[r < 0]
downside = ((down ** 2).sum() / len(r)) ** 0.5
sortino = r.mean() / downside * (252 ** 0.5)
print(f"Sortino anualizado: {sortino:.2f}")

## Drawdown y underwater
El running máximo marca el pico; el drawdown es la caída desde ahí. La zona bajo el pico es el underwater.

In [ ]:
import matplotlib.pyplot as plt

df["pico"] = df["equity"].cummax()
df["dd"] = df["equity"] / df["pico"] - 1
max_dd = df["dd"].min()
print(f"pico: {df['pico'].max():.2f}  MaxDD: {max_dd:.2%}")

fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
a1.plot(df["fecha"], df["equity"], color="#5eead4")
a1.set_ylabel("equity")
a2.fill_between(df["fecha"], df["dd"] * 100, 0, color="#f59e0b", alpha=0.5)
a2.set_ylabel("drawdown (%)")
a2.set_xlabel("fecha")
fig.tight_layout()
plt.show()

In [ ]:
# Chequeos automáticos
assert len(df) == 90, "se esperan 90 días"
assert max_dd < 0, "toda serie realista tiene drawdown"
assert abs(df["dd"].min() - max_dd) < 1e-12
assert sortino >= sharpe, "con subidas bruscas el Sortino supera al Sharpe"
print("OK: Sharpe, Sortino y MaxDD verificados")